# autoresearch-medimage - Colab Quick Start

This notebook is the fastest public trial path:
- verify that Colab can run the repo
- run a tiny smoke test
- optionally stage your own dataset and run one experiment

Use the other notebooks for heavier workflows:
- `examples/chestxray14/colab_demo.ipynb` for a single ChestXray14 benchmark demo
- `notebooks/colab_autoresearch.ipynb` for an actual multi-experiment Colab search session


## 1. Setup

This path assumes the repo is public. It downloads a public snapshot into `/content`, so no GitHub token, SSH key, or private-repo access is required.

In [ ]:
# Check whether this Colab runtime has a GPU attached
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Install uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

In [ ]:
# Download the public repo snapshot into /content
!rm -rf /content/autoresearch-medimage /content/autoresearch-medimage-main /content/repo.zip
!wget -q https://github.com/mattlungrenmd/autoresearch-medimage/archive/refs/heads/main.zip -O /content/repo.zip
!unzip -q /content/repo.zip -d /content
!mv /content/autoresearch-medimage-main /content/autoresearch-medimage
%cd /content/autoresearch-medimage

In [ ]:
# Install dependencies
!uv sync

## 2. Smoke Test

This is the fastest sanity check. It uses the built-in toy dataset and does not require any external download.

In [ ]:
!uv run prepare.py --dataset toy_binary
!uv run train.py --recipe toy_binary_smoke --time-budget 5

## 3. Bring Your Own Data

The easiest BYOD path is to upload a zip or mount Drive, then point `prepare.py --data` at that folder.

Expected image layout examples:
- classification: `class_name/image.png`
- segmentation: `images/*.png` and `masks/*.png`


In [ ]:
# Optional: mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Optional: upload a dataset archive directly from your computer
# from google.colab import files
# uploaded = files.upload()

In [ ]:
# Set this to the folder that contains your images
# Examples:
# DATA_SOURCE = '/content/drive/MyDrive/my_medical_images'
# DATA_SOURCE = '/content/my_uploaded_dataset'
DATA_SOURCE = ''

In [ ]:
if not DATA_SOURCE:
    raise ValueError('Set DATA_SOURCE before running this cell.')
!uv run prepare.py --data "{DATA_SOURCE}"

In [ ]:
# Replace the dataset name below with the one reported by prepare.py
YOUR_DATASET = ''
if not YOUR_DATASET:
    raise ValueError('Set YOUR_DATASET to the detected dataset name before running this cell.')
!uv run train.py --dataset "{YOUR_DATASET}" --save-checkpoint

In [ ]:
# Inspect the latest ledger rows
import pandas as pd
df = pd.read_csv('results.tsv', sep='\t')
df[['timestamp', 'dataset', 'backbone', 'metric', 'status']].tail(10)